# running_nb: Multilingual E-Sahulat AI

Runs the whole project on Google Colab, in order:
1. install packages
2. get the code (clone the repository, or use the uploaded project zip)
3. build the data files and run the evaluation (tunes on dev questions only, then measures dev and test)
4. build the validation pack (`validation_pack.zip`)
5. launch the web app and print a link

Run the cells top to bottom. Cell 5 prints the link to open the app.

In [ ]:
# Cell 1: install packages (the speech package is only needed for the microphone button)
!pip install -q -U streamlit sentence-transformers faster-whisper
print("Cell 1 done: packages installed")

In [ ]:
# Cell 2: get the code. Paste a GitHub/GitLab link below to clone it, or leave it empty and upload the project zip to /content
import glob, os, subprocess, zipfile

REPO_URL = ""     # for example "https://github.com/your-team/e-sahulat-ai.git"

os.chdir("/content")
if REPO_URL:
    subprocess.run(["git", "clone", REPO_URL, "/content/eSahulat_repo"], check=True)
    searchRoot = "/content/eSahulat_repo"
else:
    zipFiles = sorted(glob.glob("/content/eSahulat*.zip"), key=os.path.getmtime)
    assert zipFiles, "Upload the project zip (eSahulat....zip) to /content first, or set REPO_URL"
    with zipfile.ZipFile(zipFiles[-1]) as archive:
        archive.extractall("/content/run")
    searchRoot = "/content/run"
projectFolder = os.path.dirname(glob.glob(searchRoot + "/**/app.py", recursive=True)[0])
os.chdir(projectFolder)
print("Cell 2 done: project folder is", projectFolder)

In [ ]:
# Cell 3: build the data files, then evaluate (about 1-2 minutes; downloads the e5 model once)
!python buildData.py
!python evaluate.py
print("Cell 3 done: results are in the results/ folder")

In [ ]:
# Cell 4: build the validation pack (one row per question, retrieval table, failures, settings used)
!python makeValidationPack.py
print("Cell 4 done: validation_pack.zip created")
# optional: download it to your computer
# from google.colab import files; files.download("validation_pack.zip")

In [ ]:
# Cell 5: launch the web app and print the link
import os, re, subprocess, time, urllib.request

subprocess.run("pkill -f '[s]treamlit run'; pkill -f '[c]loudflared tunnel'", shell=True)
time.sleep(2)

cloudflaredPath = "/content/cloudflared"
if not os.path.exists(cloudflaredPath):
    print("Downloading cloudflared ...")
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cloudflaredPath)
    os.chmod(cloudflaredPath, 0o755)

subprocess.Popen("python -m streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > streamlit.log 2>&1", shell=True)
streamlitUp = False
for second in range(120):
    try:
        urllib.request.urlopen("http://localhost:8501/_stcore/health")
        streamlitUp = True
        print("Streamlit is up after", second, "seconds")
        break
    except Exception:
        time.sleep(1)

if not streamlitUp:
    print("STREAMLIT DID NOT START. Log:\n", open("streamlit.log").read()[-3000:])
else:
    subprocess.Popen(cloudflaredPath + " tunnel --url http://localhost:8501 > tunnel.log 2>&1", shell=True)
    link = None
    for second in range(90):
        time.sleep(1)
        found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
        if found:
            link = found.group(0)
            break
    if link:
        print("\nOPEN THIS LINK:", link)
    else:
        print("No tunnel link. Tunnel log:\n", open("tunnel.log").read()[-2000:])
        from google.colab import output
        output.serve_kernel_port_as_window(8501)

## Optional: push the code to GitHub
Create an empty repository on GitHub first and a personal access token with `repo` access. The token is typed into a hidden box and is not saved in the notebook.

In [ ]:
# Cell 6 (optional): push this project folder to your GitHub repository
import getpass, subprocess

repoUrl = input("Repository URL (https://github.com/you/name.git): ").strip()
token = getpass.getpass("Personal access token (hidden, not saved): ").strip()
secretUrl = repoUrl.replace("https://", "https://%s@" % token)

def git(command):
    return subprocess.run(command, shell=True, capture_output=True, text=True)

git("git init -q -b main")
git('git config user.email "team@example.com"')
git('git config user.name "E-Sahulat team"')
git("git add .")
print(git('git commit -q -m "E-Sahulat AI: multilingual evidence-grounded passport assistant"').stdout)
git("git remote remove origin")
git("git remote add origin " + secretUrl)
print(git("git push -u origin main").stderr.replace(token, "***"))
git("git remote set-url origin " + repoUrl)      # take the token out of the saved settings
print("Cell 6 done")